In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class LineRatePredictor(nn.Module):
    def __init__(self, num_features):
        super(LineRatePredictor, self).__init__()

        self.conv1 = nn.Conv1d(num_features, 32, kernel_size=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=32,
            batch_first=True
        )

        self.fc1 = nn.Linear(32, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        # x shape: (batch, time, features)

        x = x.permute(0, 2, 1)       # → (batch, features, time)

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = x.permute(0, 2, 1)       # → (batch, time, channels)

        x, _ = self.lstm(x)

        x = x[:, -1, :]              # last timestep

        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))

        return x

In [3]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

In [4]:
class SensorDataset(Dataset):

    def __init__(self, df):
        self.data = df.iloc[:,:-1].values.astype(np.float32)
        self.labels = df.iloc[:,-1].values.astype(np.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        X = self.data[idx]

        y = self.labels[idx]

        return torch.tensor(X), torch.tensor(y)
        
        

In [ ]:
# df = pd.read_csv("./sensor_dataset.csv")

# dummy_set = SensorDataset(df)
# dummy_loader = DataLoader(dummy_set, batch_size=32, shuffle=True)

df = pd.read_csv("./dataset.csv")

df_train = df.iloc[:int(len(df) * 3/4), :]
df_test = df.iloc[int(len(df) * 3/4):, :]

train_set = SensorDataset(df_train)
test_set = SensorDataset(df_test)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = DataLoader(test_set, batch_size=16)

In [7]:
model = LineRatePredictor(1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = model.to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

cuda


In [8]:
def train_model(model, train_loader, val_loader, epochs=50):

    for epoch in range(epochs):

        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            preds = model(X_batch.unsqueeze(-1)).squeeze()

            loss = criterion(preds, y_batch)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                preds = model(X_batch.unsqueeze(-1)).squeeze()

                loss = criterion(preds, y_batch)

                val_loss += loss.item()

                predicted = (preds > 0.5).float()

                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss /= len(val_loader)
        accuracy = correct / total

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {accuracy:.4f}"
        )

In [9]:
train_model(model=model, train_loader=train_loader, val_loader=test_loader)

Epoch 1/50 | Train Loss: 0.1818 | Val Loss: 0.0127 | Val Acc: 1.0000
Epoch 2/50 | Train Loss: 0.0571 | Val Loss: 0.0413 | Val Acc: 1.0000
Epoch 3/50 | Train Loss: 0.0474 | Val Loss: 0.0021 | Val Acc: 1.0000
Epoch 4/50 | Train Loss: 0.0425 | Val Loss: 0.0006 | Val Acc: 1.0000
Epoch 5/50 | Train Loss: 0.0416 | Val Loss: 0.0003 | Val Acc: 1.0000
Epoch 6/50 | Train Loss: 0.0425 | Val Loss: 0.0085 | Val Acc: 1.0000
Epoch 7/50 | Train Loss: 0.0404 | Val Loss: 0.0019 | Val Acc: 1.0000
Epoch 8/50 | Train Loss: 0.0339 | Val Loss: 0.0002 | Val Acc: 1.0000
Epoch 9/50 | Train Loss: 0.0351 | Val Loss: 0.0008 | Val Acc: 1.0000
Epoch 10/50 | Train Loss: 0.0388 | Val Loss: 0.0012 | Val Acc: 1.0000
Epoch 11/50 | Train Loss: 0.0346 | Val Loss: 0.0051 | Val Acc: 1.0000
Epoch 12/50 | Train Loss: 0.0339 | Val Loss: 0.0022 | Val Acc: 1.0000
Epoch 13/50 | Train Loss: 0.0338 | Val Loss: 0.0030 | Val Acc: 1.0000
Epoch 14/50 | Train Loss: 0.0376 | Val Loss: 0.0005 | Val Acc: 1.0000
Epoch 15/50 | Train Loss: 0.0

In [12]:
torch.save(model.state_dict(), "model3.pth")

In [13]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/model3.pth', '/content/drive/MyDrive/model3.pth')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/model3.pth'

In [ ]:
!pwd